# HPC Batch Job Submission & PyPI Publishing Guide for SearchLibrium

This notebook covers two critical workflows for SearchLibrium users:

1. **PyPI Publishing**: How to securely publish your package to PyPI with proper token authentication
2. **HPC Batch Submission**: How to submit multiple SearchLibrium model searches to an HPC cluster using PBS

By the end, you'll be able to publish packages reliably and run hundreds of model configurations in parallel on supercomputing clusters.

## Part 1: PyPI Publishing with Secure Token Authentication

### Step 1: Generate a PyPI API Token

To publish packages to PyPI, you need an API token:

1. Go to [pypi.org](https://pypi.org) and sign in to your account
2. Navigate to **Account Settings** → **API Tokens**
3. Click **Create API Token**
4. Choose **Entire account** scope (for all projects) or scope to a specific project
5. Copy the token (format: `pypi-xxx...xxx`)
6. **Store it securely** — you'll only see it once!

The token format starts with `pypi-` and looks like:
```
pypi-AgEIcHlwaS5vcmc... (long alphanumeric string)
```

### Step 2: Add Token to GitHub Repository Secrets

Now store your token as a repository secret so the CI/CD pipeline can access it:

1. Go to your GitHub repository
2. Navigate to **Settings** → **Secrets and variables** → **Actions**
3. Click **New repository secret**
4. **Name**: `PYPI_API_TOKEN` (must match exactly!)
5. **Value**: Paste your token from PyPI (e.g., `pypi-AgEIcHlwaS5vcmc...`)
6. Click **Add secret**

✅ **Important**: The secret name must exactly match what's used in your GitHub Actions workflow.

### Step 3: Build and Test Package Locally

Before pushing to GitHub, verify your package builds correctly:

```bash
# Navigate to your project directory
cd SearchLibrium

# Install build tools
pip install build wheel setuptools

# Build distribution packages (wheel + source)
python -m build

# Check the dist/ directory
ls -la dist/
```

Expected output:
```
dist/
  SearchLibrium-0.0.72-py3-none-any.whl
  SearchLibrium-0.0.72.tar.gz
```

**Verify package metadata**:
```bash
# Install twine for validation
pip install twine

# Check distribution
twine check dist/*
```

This ensures your `pyproject.toml` and `setup.py` are correctly configured before upload.

### Step 4: GitHub Actions Workflow Configuration

Your `.github/workflows/python-publish.yml` should look like this:

```yaml
name: Build and Publish to PyPI

on:
  push:
    branches: [main]
  workflow_dispatch:

jobs:
  upload-to-pypi:
    runs-on: ubuntu-latest
    permissions:
      id-token: write
      contents: write

    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with:
          python-version: "3.10"

      - name: Bump patch version
        run: |
          python - <<'EOF'
          with open("version.txt") as f:
              v = f.read().strip()
          major, minor, patch = v.split(".")
          new_version = f"{major}.{minor}.{int(patch) + 1}"
          with open("version.txt", "w") as f:
              f.write(new_version + "\n")
          print(f"Version bumped: {v} -> {new_version}")
          EOF

      - name: Commit version bump
        run: |
          git config user.name "github-actions[bot]"
          git config user.email "github-actions[bot]@users.noreply.github.com"
          git add version.txt
          git commit -m "chore: bump version [skip ci]"
          git push

      - run: |
          python -m pip install --upgrade pip
          pip install build

      - run: python -m build

      - uses: pypa/gh-action-pypi-publish@release/v1
        with:
          username: __token__
          password: ${{ secrets.PYPI_API_TOKEN }}
          attestations: false
          skip-existing: false
```

**Key points:**
- `username: __token__` — Required for API token authentication
- `password: ${{ secrets.PYPI_API_TOKEN }}` — Pulls token from repository secrets
- `skip-existing: false` — Prevents duplicate upload errors
- The version is auto-bumped with each push to main

## Part 2: HPC Batch Job Submission with PBS

### Overview: Why PBS for HPC?

When running hundreds of SearchLibrium model searches, running them locally would take weeks. HPC clusters with job schedulers like **PBS (Portable Batch System)** let you:

- Submit multiple long-running jobs simultaneously
- Each job gets dedicated CPU/memory resources
- Monitor all jobs from a single command
- Chain jobs with dependencies (Job B only runs if Job A succeeds)
- Automatically restart failed jobs

**Example**: Run 100 model configurations in parallel → typical runtime goes from weeks to hours.

### Step 5: Using the Batch Job Script

SearchLibrium includes a ready-to-use PBS batch submission script: `batch_jobs.pbs`

#### Configure Your Jobs

Edit the `batch_jobs.pbs` file to define which models to run:

```bash
# Inside batch_jobs.pbs, modify the JOBS array:
declare -a JOBS=(
    "model_search_1:search.py:04:00:00:1:32GB"
    "model_search_2:search.py:04:00:00:1:32GB"
    "model_search_3:search.py:04:00:00:1:32GB"
)
```

**Format**: `"job_name:python_script:walltime:ncpus:memory"`

- `job_name`: Unique identifier (output goes to `runs/job_name/`)
- `python_script`: Your SearchLibrium script
- `walltime`: Max runtime (HH:MM:SS)
- `ncpus`: Number of CPU cores
- `memory`: RAM needed (e.g., 32GB)

#### Examples

**Run 5 searches in parallel (4 hours each, 1 CPU, 32GB RAM)**:
```bash
declare -a JOBS=(
    "mnl_model_1:search.py:04:00:00:1:32GB"
    "mnl_model_2:search.py:04:00:00:1:32GB"
    "ml_model_1:search_mixed.py:04:00:00:1:32GB"
    "ml_model_2:search_mixed.py:04:00:00:1:32GB"
    "rrm_model:search_rrm.py:04:00:00:1:32GB"
)
```

**Run 100 jobs overnight**:
```bash
# Generate array dynamically
declare -a JOBS=()
for i in {1..100}; do
    JOBS+=("config_${i}:search.py:20:00:00:1:32GB")
done
```

### Step 6: Submit and Monitor Jobs

#### Submit Jobs in Parallel (Recommended)

All jobs run simultaneously (maximum throughput):

```bash
# From the HPC cluster login node
qsub batch_jobs.pbs

# Or explicitly:
qsub batch_jobs.pbs parallel
```

**Output**:
```
Generated job scripts...
  Generated job_model_search_1.pbs
  Generated job_model_search_2.pbs
  Generated job_model_search_3.pbs

Submitting jobs (mode: parallel)...

PARALLEL mode — all jobs run simultaneously
============================================
  model_search_1  → 12345.hpc.cluster
  model_search_2  → 12346.hpc.cluster
  model_search_3  → 12347.hpc.cluster

All 3 jobs submitted
```

#### Submit Jobs Sequentially

One job runs after the previous completes (dependencies):

```bash
qsub batch_jobs.pbs sequential
```

Job B only starts if Job A succeeds. Useful for pipeline workflows.

#### Monitor Running Jobs

```bash
# Check status of all your jobs
qstat -u $USER

# Example output:
# Job id            Name             User    Time Use S Queue
# ─────────────────────────────────────────────────────────────
# 12345.hpc         sl_model_1       me      1:23:45 R batch
# 12346.hpc         sl_model_2       me        45:32 R batch
# 12347.hpc         sl_model_3       me             0 Q batch
#                   (Q = queued, R = running, C = complete)

# View full job details
qstat -f 12345.hpc

# Watch job status in real-time (updates every 5 seconds)
watch qstat -u $USER
```

#### Monitor Job Output in Real-Time

While jobs run, watch the output files:

```bash
# Watch stdout stream as it's written
tail -f log_model_search_1.out

# Watch errors
tail -f log_model_search_1.err

# Check final output directory
ls -la runs/model_search_1/
```

#### Cancel Jobs

```bash
# Cancel a single job
qdel 12345.hpc

# Cancel all your jobs
qdel $(qstat -u $USER | grep " $USER" | awk '{print $1}')
```

### Step 7: Troubleshooting PBS Jobs

#### Job Not Starting (Stuck in Queue)

```bash
# Check why job is queued
qstat -f 12345.hpc | grep "comment"

# Common issues:
# - Not enough memory available → reduce requested memory
# - Exceeds walltime limit → increase walltime
# - Module not available → check module load in script
```

#### Out of Memory (OOM) Error

Job exits with exit code 127 or similar:

```bash
# Check error log
tail -50 log_model_search_1.err

# Solutions:
# 1. Increase memory in JOBS array: "model_1:search.py:04:00:00:1:64GB"
# 2. Reduce simulation draws in SearchLibrium config
# 3. Split into smaller datasets
```

#### Job Timeout

Walltime exceeded before job completes:

```bash
# Check how long job actually took
tail log_model_search_1.out | grep "Finished"

# Increase walltime:
"model_1:search.py:10:00:00:1:32GB"  # Changed from 04:00:00 to 10:00:00
```

#### Module Load Failures

```bash
# Check available modules
module avail

# If conda not available:
module load Anaconda3/2024.02-1
conda env list
```

### Step 8: Verify PyPI Upload Success

After pushing code to main branch:

```bash
# Watch GitHub Actions workflow
# https://github.com/YOUR-ORG/SearchLibrium/actions

# Once workflow completes, verify package is on PyPI
pip install --upgrade SearchLibrium

# Check installed version
python -c "import SearchLibrium; print(SearchLibrium.__version__)"

# Test import
python -c "from SearchLibrium import Parameters, call_siman; print('✓ Import successful')"
```

**Expected**:
```
✓ Import successful
```

#### If Upload Failed

1. **Check GitHub Actions logs**:
   - Go to Actions → Build and Publish to PyPI
   - Click the failed run
   - Review logs for token/build errors

2. **Common causes**:
   - Token expired or incorrect name
   - Package name already exists on PyPI
   - Version already uploaded
   - Missing dependencies in pyproject.toml

3. **Fix and retry**:
   ```bash
   # Push again after fix
   git add .
   git commit -m "Fix: resolve PyPI upload issue"
   git push
   ```

## Quick Reference

### PyPI Upload Checklist

- [ ] Generated API token on pypi.org
- [ ] Token starts with `pypi-`
- [ ] Stored token as `PYPI_API_TOKEN` in GitHub Secrets
- [ ] GitHub Actions workflow has `username: __token__` and `password: ${{ secrets.PYPI_API_TOKEN }}`
- [ ] Pushed code to main branch
- [ ] Workflow completed successfully
- [ ] Package appears on pypi.org
- [ ] `pip install SearchLibrium` works

### PBS Batch Job Checklist

- [ ] Edit JOBS array in `batch_jobs.pbs` with your models
- [ ] Test scripts locally first
- [ ] Check cluster's conda environment name (e.g., `venvnewg`)
- [ ] Run: `qsub batch_jobs.pbs` (parallel) or `qsub batch_jobs.pbs sequential`
- [ ] Monitor: `qstat -u $USER`
- [ ] Watch logs: `tail -f log_<jobname>.out`
- [ ] Check results in `runs/<jobname>/`

### Common Commands

| Task | Command |
|------|---------|
| Submit parallel jobs | `qsub batch_jobs.pbs` |
| Submit sequential jobs | `qsub batch_jobs.pbs sequential` |
| Check all your jobs | `qstat -u $USER` |
| Check one job | `qstat -f 12345.hpc` |
| Watch output | `tail -f log_model_1.out` |
| Cancel job | `qdel 12345.hpc` |
| Build package | `python -m build` |
| Check build | `twine check dist/*` |
| Install from PyPI | `pip install SearchLibrium` |